
# 07: All-Sky RR Lyrae Reddening Map

This notebook extends the empirical period-color calibration from `05.ipynb` and the extinction calculation from `06.ipynb` into a sky map. The main product is an Aitoff projection of the RR Lyrae color excess

$$
E(G_{\mathrm{BP}} - G_{\mathrm{RP}}),
$$

plotted against Galactic longitude and latitude for the shared RRab/RRc Gaia catalog. I first show the full uncut map, then apply explicit quality cuts to remove obvious photometric outliers while preserving broad sky coverage.

The quality strategy is intentionally simple and tied to Gaia photometry: require usable BP and RP signal-to-noise, reject stars with anomalous BP/RP excess, and remove stars whose propagated reddening uncertainty is too large for the map to be visually stable.


The PDF wording describes coloring the map by $A_G$. In the current notebook I keep the primary map in $E(G_{\mathrm{BP}}-G_{\mathrm{RP}})$ instead, because that is the directly inferred empirical quantity from `06.ipynb`. With the fixed conversion $A_G = R_G E(G_{\mathrm{BP}}-G_{\mathrm{RP}})$ and $R_G = 2.0$, the two maps differ only by a uniform multiplicative scale, so the sky morphology and the quality-cut logic are identical.

In [ ]:

from pathlib import Path
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
from astropy import table

from ugdatalab import (
    build_quality_components,
    build_stage_summary,
    build_criterion_failure_table,
    build_reddening_quality_mask,
    load_table_npz,
    plot_aitoff_reddening_map,
    plot_quality_diagnostics,
    plot_reddening_distribution,
)

PC_COMPARISON_PATH = Path("rrlyrae_optical_pc_comparison_data.npz")
FULL_CATALOG_PATH = Path("rrlyrae_rrab_rrc_full_catalog.npz")
EXTINCTION_CATALOG_PATH = Path("rrlyrae_extinction_catalog.npz")
R_G = 2.0

RR_CLASSES = ("RRab", "RRc")
MIN_BP_SNR = 5.0
MIN_RP_SNR = 5.0
APPLY_BP_RP_EXCESS_CUT = True
MAX_SIGMA_E = 0.15
MIN_EBPRP = 0.0           # physically motivated: dust only reddens
MAX_EBPRP = 10.0          # physical upper ceiling (mag, Gaia BP-RP)
MIN_REDDENING_SNR = None  # set to float (e.g. 1.0) to enable
MIN_RETAINED_STARS = 60_000
DIAGNOSTIC_SAMPLE_SIZE = 25_000
RNG_SEED = 7



## Load the Saved Period-Color Fits and Full RRab/RRc Catalog

As in `06.ipynb`, this notebook relies on the two local `.npz` handoff files produced earlier in the lab:

- `rrlyrae_optical_pc_comparison_data.npz` with the class-specific RRab and RRc period-color summaries from `05.ipynb`.
- `rrlyrae_rrab_rrc_full_catalog.npz` with the shared Gaia RRab/RRc cache.

Those summaries are converted into the input format expected by `compute_period_color_extinction(...)`, which evaluates the intrinsic color relation for each star and appends `E_bprp`, `A_G_calc`, and `sigma_E`.


In [ ]:

try:
    rrlyrae_extinction = load_table_npz(EXTINCTION_CATALOG_PATH)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Required artifact rrlyrae_extinction_catalog.npz not found in labs/01. '
        'Run 06.ipynb to regenerate it before executing this notebook.'
    ) from exc

finite_empirical = np.isfinite(np.asarray(rrlyrae_extinction['E_bprp'], dtype=float))
full_catalog_summary = table.Table(
    rows=[
        {
            'class': rr_class,
            'N_catalog': int((rrlyrae_extinction["best_classification"] == rr_class).sum()),
            'N_finite_E': int((finite_empirical & (rrlyrae_extinction["best_classification"] == rr_class)).sum()),
        }
        for rr_class in RR_CLASSES
    ]
)

print(f"Shared RRab/RRc catalog: {len(rrlyrae_extinction):,} stars")
display(full_catalog_summary)


## Quality-Cut Strategy

The uncut map contains genuine large-scale dust structure, but it also contains isolated points whose inferred reddening is implausibly different from neighboring stars. To remove those failures without destroying the sky coverage, I apply four requirements:

1. finite empirical reddening quantities and Gaia color,
2. `phot_bp_mean_flux_over_error > 5`,
3. `phot_rp_mean_flux_over_error > 5`,
4. Gaia BP/RP excess inside the commonly used color-dependent envelope
   $$1.0 + 0.015(BP-RP)^2 < C < 1.3 + 0.06(BP-RP)^2,$$
5. propagated reddening uncertainty `sigma_E <= 0.15` mag.

The BP/RP signal-to-noise and excess cuts target photometry that is likely inconsistent, blended, or otherwise unreliable in Gaia's blue and red photometers. The final `sigma_E` threshold is chosen after inspecting the uncut map: it removes the visually worst star-by-star outliers while still leaving far more than the required 60,000 stars.


In [ ]:

components = build_quality_components(
    rrlyrae_extinction,
    min_bp_snr=MIN_BP_SNR,
    min_rp_snr=MIN_RP_SNR,
    max_sigma_e=MAX_SIGMA_E,
    min_ebprp=MIN_EBPRP,
    max_ebprp=MAX_EBPRP,
)

adopted_mask = build_reddening_quality_mask(
    rrlyrae_extinction,
    min_bp_snr=MIN_BP_SNR,
    min_rp_snr=MIN_RP_SNR,
    apply_bp_rp_excess_cut=APPLY_BP_RP_EXCESS_CUT,
    max_sigma_E=MAX_SIGMA_E,
    min_ebprp=MIN_EBPRP,
    max_ebprp=MAX_EBPRP,
    min_reddening_snr=MIN_REDDENING_SNR,
)
if not np.array_equal(adopted_mask, components['adopted']):
    raise AssertionError('Notebook-local mask components disagreed with build_reddening_quality_mask().')

removed_mask = components['finite'] & ~adopted_mask
e_valid = np.asarray(rrlyrae_extinction['E_bprp'], dtype=float)[components['finite']]
color_vmin = float(min(0.0, np.nanpercentile(e_valid, 0.5)))
color_vmax = float(np.nanpercentile(e_valid, 99.5))

stage_summary = build_stage_summary(rrlyrae_extinction, components)
criterion_failures = build_criterion_failure_table(rrlyrae_extinction, components)
final_summary = table.Table(
    rows=[
        {
            'N_retained': int(adopted_mask.sum()),
            'N_removed': int(removed_mask.sum()),
            'RRab_retained': int(np.count_nonzero(adopted_mask & (rrlyrae_extinction["best_classification"] == 'RRab'))),
            'RRc_retained': int(np.count_nonzero(adopted_mask & (rrlyrae_extinction["best_classification"] == 'RRc'))),
            'fraction_of_full': round(float(adopted_mask.sum()) / len(rrlyrae_extinction), 4),
            'fraction_of_finite': round(float(adopted_mask.sum()) / int(components['finite'].sum()), 4),
        }
    ]
)

if int(adopted_mask.sum()) < MIN_RETAINED_STARS:
    raise RuntimeError(
        f'Adopted cut retains only {int(adopted_mask.sum()):,} stars, below the required {MIN_RETAINED_STARS:,}.'
    )

display(stage_summary)
display(criterion_failures)
display(final_summary)
print(f'Adopted quality mask retains {int(adopted_mask.sum()):,} stars.')



## Uncut Aitoff Map

The first map keeps every RRab or RRc star with a finite empirical color excess. The broad dust structure is already obvious: reddening increases strongly toward the Galactic plane and the inner Galaxy. At the same time, there are isolated points whose colors are hard to reconcile with their local surroundings, which is exactly the failure mode the quality cuts are meant to suppress.


In [ ]:

fig_uncut, ax_uncut = plot_aitoff_reddening_map(
    rrlyrae_extinction,
    components['finite'],
    title='Uncut RR Lyrae reddening map',
    vmin=color_vmin,
    vmax=color_vmax,
)
plt.show()



## Why These Cuts?

The map-level outliers are not random in origin. Weak BP or RP fluxes inflate the observed color uncertainty, while large BP/RP excess often marks blended or otherwise problematic Gaia photometry. The two-panel diagnostic below makes those two failure channels explicit: the adopted `sigma_E` cut trims the long high-uncertainty tail, and the BP/RP excess envelope removes stars whose BP+RP flux is inconsistent with the broad-band `G` measurement.


In [ ]:

fig_diag, axes_diag = plot_quality_diagnostics(
    rrlyrae_extinction,
    components,
    sample_size=DIAGNOSTIC_SAMPLE_SIZE,
    seed=RNG_SEED,
    save=True,
)
plt.show()


In [ ]:
# Physical reddening value diagnostics — applied to the stage-5 (sigma_E-passed) sample
stage5_mask = (
    components['finite']
    & components['bp_snr']
    & components['rp_snr']
    & components['bp_rp_excess']
    & components['sigma_e']
)
e_stage5 = np.asarray(rrlyrae_extinction['E_bprp'], dtype=float)[stage5_mask]

n_negative = int((e_stage5 < 0).sum())
n_extreme = int((e_stage5 > MAX_EBPRP).sum())
print(f"Stars with E(BP-RP) < 0 in stage-5 sample:        {n_negative:,}  ({100*n_negative/len(e_stage5):.2f}%)")
print(f"Stars with E(BP-RP) > {MAX_EBPRP:.1f} in stage-5 sample:  {n_extreme:,}  ({100*n_extreme/len(e_stage5):.2f}%)")
print(f"Stage-5 sample size:                               {len(e_stage5):,}")

fig_phys, ax_phys = plot_reddening_distribution(e_stage5, min_ebprp=MIN_EBPRP, max_ebprp=MAX_EBPRP, save=True)
plt.show()



## Cleaned Aitoff Map After the Adopted Quality Cuts

After applying the full quality mask, the isolated extreme points are much less prominent, but the large-scale Galactic dust structure remains. The retained sample is still comfortably above the assignment floor, so the price of cleaning the map is modest compared with the gain in visual coherence.


In [ ]:

fig_cut, ax_cut = plot_aitoff_reddening_map(
    rrlyrae_extinction,
    adopted_mask,
    title='Quality-cut RR Lyrae reddening map',
    vmin=color_vmin,
    vmax=color_vmax,
    save="fig_reddening_map.pdf",
)
plt.show()



## Removed Objects

Plotting only the rejected stars makes the action of the cuts easier to interpret. The removed set is not uniformly random: it concentrates in crowded, dusty, and low-latitude regions where Gaia BP/RP photometry is more easily perturbed, but there are also scattered high-latitude failures that look like individual photometric outliers rather than coherent dust structure.


In [ ]:

fig_removed, ax_removed = plot_aitoff_reddening_map(
    rrlyrae_extinction,
    removed_mask,
    title='Stars removed by the adopted quality mask',
    vmin=color_vmin,
    vmax=color_vmax,
    alpha=0.28,
    size=3.0,
    save_name="fig_removed_stars_map.pdf",
)
plt.show()


## Analysis and Discussion

The cleaned map behaves the way an empirical reddening map should. The strongest color excess appears close to the Galactic plane and toward the inner Galaxy, where the dust column is largest. That broad morphology is consistent with the established all-sky dust-map picture from infrared-emission-based work and its later recalibration ([Schlegel, Finkbeiner, & Davis 1998](https://doi.org/10.1086/305772); [Schlafly & Finkbeiner 2011](https://ui.adsabs.harvard.edu/abs/2011ApJ...737..103S/abstract)).

### Why E(BP-RP) ≥ 0 Is Physically Necessary

Dust extinction is an absorptive and scattering process that is always positive — photons are removed from the beam, never added. The Cardelli, Clayton, & Mathis (1989) and Fitzpatrick (1999) extinction laws parametrize this as a strictly positive, wavelength-dependent opacity, so the true color excess $E(G_\mathrm{BP} - G_\mathrm{RP})$ cannot be negative for any sightline containing foreground dust. Measured negative values therefore indicate one of three failure modes: (1) the star's measured color is bluer than the intrinsic-color relation predicts due to photometric noise, (2) the period-color calibration is extrapolated outside its training range, or (3) the source is an unresolved blend that biases the BP and RP fluxes in opposite directions. Cutting at $E \geq 0$ removes this unphysical tail cleanly without any tuning of the underlying relations. The upper ceiling at $E \leq 3.0$ mag serves a complementary role: no RR Lyrae in the Milky Way halo or disk should suffer more than roughly 3 mag of $E(G_\mathrm{BP}-G_\mathrm{RP})$ reddening, so values above this threshold point to contaminated photometry rather than to extreme dust columns.

### Large-Scale Structure in the Reddening Map

The map shows the dust concentrated in the Galactic plane, with a sharp gradient from low- to high-latitude sightlines. The inner Galaxy region ($|l| < 30°$) is the reddest part of the map, consistent with the high dust column toward the Galactic bar and bulge. At positive latitudes the Large Magellanic Cloud and Small Magellanic Cloud sightlines contribute an elevated reddening "island" in the southern hemisphere at $(l, b) \approx (280°, -33°)$ and $(302°, -44°)$ respectively. These are genuine astrophysical signatures rather than artifacts.

The concentration of RR Lyrae toward the inner Galaxy is not purely a dust effect. Pietrukowicz et al. (2015) showed that the Galactic bulge hosts an overdensity of OGLE RRab stars tracing the old ($> 10$ Gyr) stellar population in the bar, and that same population appears in the Gaia-based catalog used here. The bulge excess therefore shows up simultaneously as a reddening peak (because the inner Galaxy is dusty) and as a tracer-number excess (because the bulge stellar density is high).

### Non-Uniform RR Lyrae Sky Distribution

The RR Lyrae sky distribution is non-uniform for two separate reasons. First, the underlying stellar populations are not uniform: Gaia DR3 RR Lyrae trace the Galactic halo, bulge, Magellanic Clouds, globular clusters, and dwarf satellites, so real substructure is built into the map from the start ([Clementini et al. 2023](https://www.aanda.org/articles/aa/full_html/2023/06/aa43964-22/aa43964-22.html)). Second, the observed Gaia sample is not spatially uniform because time sampling, crowding, and variability-processing completeness vary across the sky, so the catalog itself reflects the Gaia scanning law and pipeline selection function rather than a simple uniform tracer population ([Eyer et al. 2023](https://www.aanda.org/articles/aa/full_html/2023/06/aa44242-22/aa44242-22.html)).

The obvious isolated failures in the uncut map are also expected. Gaia BP/RP photometry can be degraded by crowding, blends, background issues, and other color-dependent photometric inconsistencies, which is why the BP/RP excess factor is useful as a diagnostic rather than as a universal one-size-fits-all cut ([Riello et al. 2021](https://www.aanda.org/articles/aa/full_html/2021/05/aa39587-20/aa39587-20.html)).

The direct answer to the assignment question is therefore: the RR Lyrae distribution is non-uniform because it combines real Milky Way and Local Group structure with Gaia's sky-dependent observational completeness. The cleaned reddening map still inherits both effects, but after the quality cuts the dominant large-scale features are much more plausibly astrophysical rather than single-star photometric failures.

### References

- Cardelli, J. A., Clayton, G. C., & Mathis, J. S. (1989), "The relationship between infrared, optical, and ultraviolet extinction," *ApJ*, 345, 245. https://doi.org/10.1086/167900
- Clementini, G., et al. (2023), "Gaia Data Release 3 - Specific processing and validation of all-sky RR Lyrae and Cepheid stars: The RR Lyrae sample," *A&A*, 674, A18. https://www.aanda.org/articles/aa/full_html/2023/06/aa43964-22/aa43964-22.html
- Eyer, L., et al. (2023), "Gaia Data Release 3 - Summary of the variability processing and analysis," *A&A*, 674, A13. https://www.aanda.org/articles/aa/full_html/2023/06/aa44242-22/aa44242-22.html
- Fitzpatrick, E. L. (1999), "Correcting for the effects of interstellar extinction," *PASP*, 111, 63. https://doi.org/10.1086/316293
- Pietrukowicz, P., et al. (2015), "Deciphering the 3D structure of the Old Milky Way Bulge in OGLE RR Lyrae stars," *ApJ*, 811, 113. https://doi.org/10.1088/0004-637X/811/2/113
- Riello, M., et al. (2021), "Gaia Early Data Release 3 - Photometric content and validation," *A&A*, 649, A3. https://www.aanda.org/articles/aa/full_html/2021/05/aa39587-20/aa39587-20.html
- Schlegel, D. J., Finkbeiner, D. P., & Davis, M. (1998), "Maps of Dust Infrared Emission for Use in Estimation of Reddening and Cosmic Microwave Background Radiation Foregrounds," *ApJ*, 500, 525. https://doi.org/10.1086/305772
- Schlafly, E. F., & Finkbeiner, D. P. (2011), "Measuring Reddening with Sloan Digital Sky Survey Stellar Spectra and Recalibrating SFD," *ApJ*, 737, 103. https://ui.adsabs.harvard.edu/abs/2011ApJ...737..103S/abstract


Because the notebook adopts a fixed $R_G = 2.0$, exactly the same cleaned sky pattern would appear in an $A_G$-colored map after a simple factor-of-two rescaling. Keeping the map in empirical color excess just makes the connection to the period-color calibration from `05.ipynb` and the extinction calculation from `06.ipynb` more transparent.


# 08: Comparing the RR Lyrae Reddening Map to SFD

This notebook compares the cleaned RR Lyrae reddening map from `07.ipynb` to the classic Schlegel, Finkbeiner, & Davis (1998; SFD) dust map. The SFD product is sampled at the same Galactic coordinates as the RR Lyrae stars, but the comparison is kept in native units:

- the empirical RR Lyrae map stays in `E(G_BP - G_RP)`,
- the SFD map stays in `E(B-V)`.

That choice makes the notebook a comparison of morphology and sightline-to-sightline structure rather than an artificial unit-conversion exercise.


In [ ]:
from pathlib import Path
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
from astropy import table

from ugdatalab import (
    cache_stable,
    binned_median_trend,
    build_quality_components,
    build_reddening_quality_mask,
    load_table_npz,
    plot_aitoff_sfd_map,
    plot_aitoff_value_map,
    plot_reddening_distribution,
    plot_regime_decomposition,
    plot_sfd_all_sky_hexbin,
    plot_sfd_empirical_hexbin_comparison,
    rank_spearman,
    sample_sfd_ebv,
    subset_row,
)

PC_COMPARISON_PATH = Path("rrlyrae_optical_pc_comparison_data.npz")
FULL_CATALOG_PATH = Path("rrlyrae_rrab_rrc_full_catalog.npz")
EXTINCTION_CATALOG_PATH = Path("rrlyrae_extinction_catalog.npz")
DUSTMAPS_DATA_DIR = Path('.dustmaps-data')
R_G = 2.0

RR_CLASSES = ("RRab", "RRc")
MIN_BP_SNR = 5.0
MIN_RP_SNR = 5.0
MAX_SIGMA_E = 0.15
MIN_EBPRP = 0.0           # physically motivated: dust only reddens
MAX_EBPRP = 10.0          # physical upper ceiling (mag, Gaia BP-RP)
MIN_REDDENING_SNR = None  # set to float (e.g. 1.0) to enable
MIN_RETAINED_STARS = 60_000
PLANE_LATITUDE_MAX = 15.0
HIGH_LATITUDE_MIN = 30.0
BIN_COUNT = 18



## Rebuild the Same Cleaned RR Lyrae Sample Used in `07.ipynb`

To keep the SFD comparison aligned with the previous notebook, I recompute the empirical reddening table and then apply the same adopted quality mask:

- `phot_bp_mean_flux_over_error > 5`
- `phot_rp_mean_flux_over_error > 5`
- Gaia BP/RP excess inside the standard color-dependent envelope
- `sigma_E <= 0.15`

This ensures that the SFD comparison is made against the cleaned map rather than against the noisier unfiltered RR Lyrae sample.


In [ ]:

try:
    rrlyrae_extinction = load_table_npz(EXTINCTION_CATALOG_PATH)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Required artifact rrlyrae_extinction_catalog.npz not found in labs/01. '
        'Run 06.ipynb to regenerate it before executing this notebook.'
    ) from exc

adopted_mask = build_reddening_quality_mask(
    rrlyrae_extinction,
    min_bp_snr=MIN_BP_SNR,
    min_rp_snr=MIN_RP_SNR,
    apply_bp_rp_excess_cut=True,
    max_sigma_E=MAX_SIGMA_E,
    min_ebprp=MIN_EBPRP,
    max_ebprp=MAX_EBPRP,
    min_reddening_snr=MIN_REDDENING_SNR,
)

if int(adopted_mask.sum()) < MIN_RETAINED_STARS:
    raise RuntimeError(
        f'Adopted cut retains only {int(adopted_mask.sum()):,} stars, below the required {MIN_RETAINED_STARS:,}.'
    )

rrlyrae_clean = rrlyrae_extinction[adopted_mask].copy()
cleaned_summary = table.Table(
    rows=[
        {
            'class': rr_class,
            'N_retained': int(np.count_nonzero((rrlyrae_clean["best_classification"] == rr_class))),
        }
        for rr_class in RR_CLASSES
    ]
)

print(f'Quality-cut RR Lyrae sample: {len(rrlyrae_clean):,} stars')
display(cleaned_summary)


## Guarded SFD Setup and Cached Sampling

The `dustmaps` package is already available in the environment, but the SFD FITS files may not be. This cell does three things:

1. points `dustmaps` to a notebook-local data directory,
2. downloads only the SFD map if it is missing,
3. caches the sampled `E(B-V)` values for the retained RR Lyrae positions through the shared joblib cache helper.

The raw dustmaps files and the sampled sightline results are therefore cached separately: the FITS files live in `.dustmaps-data/`, while the sampled arrays live in `.joblib-cache/` through `cache_stable`.


In [ ]:

try:
    from dustmaps.config import config as dustmaps_config
    import dustmaps.sfd
    from dustmaps.sfd import SFDQuery
except ImportError as exc:
    raise RuntimeError(
        'dustmaps is required for the SFD comparison notebook. '
        'Install the optional dust dependency and rerun this notebook.'
    ) from exc

DUSTMAPS_DATA_DIR.mkdir(parents=True, exist_ok=True)
dustmaps_config['data_dir'] = str(DUSTMAPS_DATA_DIR.resolve())


def ensure_sfd_available():
    try:
        SFDQuery()
        return 'available'
    except FileNotFoundError:
        print('SFD files were not found locally; downloading the SFD map...')
        try:
            dustmaps.sfd.fetch()
        except Exception as exc:
            raise RuntimeError(
                'SFD files were missing and the notebook could not download them. '
                'Check network access, rerun this cell, and then continue.'
            ) from exc
        SFDQuery()
        return 'downloaded'


@cache_stable(module='ugdatalab.dust')
def sample_sfd_cached(l_deg, b_deg):
    coords = table.Table(
        {
            'l': np.asarray(l_deg, dtype=float),
            'b': np.asarray(b_deg, dtype=float),
        }
    )
    return sample_sfd_ebv(coords)


sfd_status = ensure_sfd_available()
rrlyrae_clean['sfd_ebv'] = sample_sfd_cached(
    np.asarray(rrlyrae_clean['l'], dtype=float),
    np.asarray(rrlyrae_clean['b'], dtype=float),
)

sfd_finite_mask = np.isfinite(np.asarray(rrlyrae_clean['sfd_ebv'], dtype=float))
if not np.all(sfd_finite_mask):
    rrlyrae_clean = rrlyrae_clean[sfd_finite_mask].copy()

sfd_summary = table.Table(
    rows=[
        {
            'N_retained_rrlyrae': len(rrlyrae_clean),
            'RRab_retained': int(np.count_nonzero((rrlyrae_clean["best_classification"] == 'RRab'))),
            'RRc_retained': int(np.count_nonzero((rrlyrae_clean["best_classification"] == 'RRc'))),
            'finite_sfd_rows': int(np.count_nonzero(sfd_finite_mask)),
            'sfd_status': sfd_status,
        }
    ]
)
display(sfd_summary)


In [ ]:

# SFD E(B-V) value diagnostics
sfd_arr = np.asarray(rrlyrae_clean['sfd_ebv'], dtype=float)
print(f"Negative SFD E(B-V): {int((sfd_arr < 0).sum())} stars")
print(f"SFD E(B-V) > 5.0:    {int((sfd_arr > 5.0).sum())} stars")
print(f"Median SFD E(B-V):   {np.nanmedian(sfd_arr):.4f} mag")
print(f"Total stars with SFD values: {int(np.isfinite(sfd_arr).sum())}")



## Plotting and Comparison Helpers

The map panels use the same Aitoff projection style as `07.ipynb`, but generalized so either the empirical RR Lyrae reddening or the sampled SFD values can be plotted. I also compute a simple Spearman rank correlation and binned medians for the matched-sightline comparison without forcing the two maps into the same units.



## Side-by-Side Aitoff Maps at the Same Positions

The two maps below use the same cleaned RR Lyrae sightlines. The left panel is the empirical RR Lyrae color excess in Gaia colors, while the right panel is the SFD `E(B-V)` sampled at those same coordinates. Because the units differ, each panel keeps its own color scale.


In [ ]:

empirical = np.asarray(rrlyrae_clean['E_bprp'], dtype=float)
sfd_ebv = np.asarray(rrlyrae_clean['sfd_ebv'], dtype=float)
l_deg = np.asarray(rrlyrae_clean['l'], dtype=float)
b_deg = np.asarray(rrlyrae_clean['b'], dtype=float)

empirical_vmin = float(min(0.0, np.nanpercentile(empirical, 0.5)))
empirical_vmax = float(np.nanpercentile(empirical, 99.5))
sfd_vmin = float(max(0.0, np.nanpercentile(sfd_ebv, 0.5)))
sfd_vmax = float(np.nanpercentile(sfd_ebv, 99.5))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=180, subplot_kw={'projection': 'aitoff'})
sc_empirical, label_empirical = plot_aitoff_value_map(
    axes[0],
    l_deg,
    b_deg,
    empirical,
    title='RR Lyrae empirical $E(G_{\\mathrm{BP}}-G_{\\mathrm{RP}})$',
    vmin=empirical_vmin,
    vmax=empirical_vmax,
    colorbar_label=r'$E(G_{\mathrm{BP}} - G_{\mathrm{RP}})$ [mag]',
)
sc_sfd, label_sfd = plot_aitoff_value_map(
    axes[1],
    l_deg,
    b_deg,
    sfd_ebv,
    title='Sampled SFD $E(B-V)$',
    vmin=sfd_vmin,
    vmax=sfd_vmax,
    colorbar_label=r'SFD $E(B-V)$ [mag]',
    cmap='viridis',
)

fig.colorbar(sc_empirical, ax=axes[0], orientation='horizontal', pad=0.08, shrink=0.9, label=label_empirical)
fig.colorbar(sc_sfd, ax=axes[1], orientation='horizontal', pad=0.08, shrink=0.9, label=label_sfd)
fig.suptitle(f'Matched-sightline comparison on {len(rrlyrae_clean):,} cleaned RR Lyrae stars', y=1.02)
plt.tight_layout()
plt.show()


## Standalone SFD Aitoff Map

The sampled SFD $E(B-V)$ plotted in the same Aitoff projection as the empirical reddening map. Because SFD integrates the full dust column to infinity, the plane structure is more pronounced and extends to higher values than the distance-limited stellar measurements.

In [ ]:
all_mask = np.ones(len(rrlyrae_clean), dtype=bool)
sfd_vmin = float(np.nanpercentile(sfd_ebv, 0.5))
sfd_vmax = float(np.nanpercentile(sfd_ebv, 99.5))

fig_sfd_map, ax_sfd_map = plot_aitoff_sfd_map(
    rrlyrae_clean,
    all_mask,
    title=r'SFD $E(B-V)$ sampled at RR Lyrae positions',
    vmin=sfd_vmin,
    vmax=sfd_vmax,
    save='fig_sfd_map.pdf',
)
plt.show()


## Quantitative Matched-Sightline Comparison

A point-by-point agreement is not expected, because the maps are not measuring the same physical quantity. Even so, the matched-sightline comparison should still show that higher-SFD regions correspond, on average, to higher empirical RR Lyrae color excess. The table and density plot below summarize that trend.


In [ ]:
abs_b = np.abs(np.asarray(rrlyrae_clean['b'], dtype=float))
all_mask = np.ones(len(rrlyrae_clean), dtype=bool)
plane_mask = abs_b < PLANE_LATITUDE_MAX
intermediate_mask = (abs_b >= PLANE_LATITUDE_MAX) & (abs_b < HIGH_LATITUDE_MIN)
high_lat_mask = abs_b >= HIGH_LATITUDE_MIN

comparison_summary = table.Table(
    rows=[
        subset_row('all cleaned stars', rrlyrae_clean, all_mask),
        subset_row(r'|b| < 15 deg', rrlyrae_clean, plane_mask),
        subset_row(r'15 <= |b| < 30 deg', rrlyrae_clean, intermediate_mask),
        subset_row(r'|b| >= 30 deg', rrlyrae_clean, high_lat_mask),
    ]
)
display(comparison_summary)

fig, ax = plot_sfd_all_sky_hexbin(rrlyrae_clean, save='fig_sfd_comparison.pdf')
plt.show()

In [ ]:
# Latitude-binned Pearson R² table
from scipy.stats import pearsonr

abs_b_all = np.abs(np.asarray(rrlyrae_clean['b'], dtype=float))
empirical_all = np.asarray(rrlyrae_clean['E_bprp'], dtype=float)
sfd_all = np.asarray(rrlyrae_clean['sfd_ebv'], dtype=float)

bin_edges = np.arange(0, 91, 10)  # 0-10, 10-20, ..., 80-90
lat_rows = []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    sel = (abs_b_all >= lo) & (abs_b_all < hi)
    n = int(sel.sum())
    if n >= 10:
        r, pval = pearsonr(sfd_all[sel], empirical_all[sel])
        r2 = r**2
    else:
        r2, pval = float('nan'), float('nan')
    lat_rows.append({
        '|b| bin': f'{lo:.0f}–{hi:.0f}°',
        'N_stars': n,
        'R2': round(float(r2), 4),
        'p_value': f'{pval:.2e}' if not np.isnan(pval) else 'N/A',
    })

lat_table = table.Table(rows=lat_rows)
print("Latitude-binned Pearson R² between SFD E(B-V) and RR Lyrae E(BP-RP):")
display(lat_table)

## Regime Decomposition: Similar-Scale vs. Large-SFD Sightlines

The full-sample density plot measures the global monotonic agreement, but the physical behaviour is better understood by splitting on the *magnitude* of SFD E(B-V) rather than on sky position. Two regimes are examined:

1. **Similar scale** ($E(B-V)_\mathrm{SFD} \leq 2$ mag): SFD and empirical are in the same dynamic range. A linear trend with slope $\approx R_{BP-RP}$ is expected.
2. **Large SFD** ($E(B-V)_\mathrm{SFD} > 10$ mag): the SFD column greatly exceeds the empirical value, exposing the distance-limited nature of the RR Lyrae measurement.

In [ ]:
SIMILAR_SCALE_MAX = 2.0   # E(B-V) [mag]: upper edge of "comparable" regime
LARGE_SFD_MIN = 10.0      # E(B-V) [mag]: lower edge of "large SFD" regime

finite_mask = np.isfinite(sfd_ebv) & np.isfinite(empirical)
similar_mask = finite_mask & (sfd_ebv <= SIMILAR_SCALE_MAX)
large_mask   = finite_mask & (sfd_ebv >  LARGE_SFD_MIN)

print(f"Similar-scale regime (SFD ≤ {SIMILAR_SCALE_MAX} mag): {similar_mask.sum():,} stars")
print(f"Large-SFD regime     (SFD > {LARGE_SFD_MIN} mag):  {large_mask.sum():,} stars")
print(f"Transition (2 < SFD ≤ 10 mag):  {(finite_mask & (sfd_ebv > SIMILAR_SCALE_MAX) & (sfd_ebv <= LARGE_SFD_MIN)).sum():,} stars")

In [ ]:
# --- linear fit in the similar-scale regime ---
from scipy.stats import pearsonr

x_sim = sfd_ebv[similar_mask]
y_sim = empirical[similar_mask]
slope, intercept = np.polyfit(x_sim, y_sim, 1)
r_sim, _ = pearsonr(x_sim, y_sim)
r2_sim = r_sim**2

fig, axes = plot_regime_decomposition(
    sfd_ebv, empirical,
    similar_mask=similar_mask,
    large_mask=large_mask,
    slope=slope,
    intercept=intercept,
    r2_sim=r2_sim,
    similar_scale_max=SIMILAR_SCALE_MAX,
    large_sfd_min=LARGE_SFD_MIN,
    save='fig_regime_decomposition.pdf',
)
plt.show()

In [ ]:
from IPython.display import Markdown

similar_median_empirical = float(np.nanmedian(empirical[similar_mask]))
similar_median_sfd       = float(np.nanmedian(sfd_ebv[similar_mask]))
large_median_empirical   = float(np.nanmedian(empirical[large_mask])) if large_mask.sum() > 0 else float('nan')
large_median_sfd         = float(np.nanmedian(sfd_ebv[large_mask]))  if large_mask.sum() > 0 else float('nan')

regime_discussion = rf"""
### Similar-Scale Regime (SFD $\leq$ {SIMILAR_SCALE_MAX} mag)

In the {similar_mask.sum():,}-star similar-scale sample (median SFD = {similar_median_sfd:.3f} mag, median empirical = {similar_median_empirical:.3f} mag), the two dust tracers are in the same dynamic range and the Pearson $R^2 = {r2_sim:.3f}$. The linear fit slope is **{slope:.3f}**, close to the expected Gaia-band conversion factor $R_{{BP-RP}} = E(G_{{BP}}-G_{{RP}})/E(B-V) \approx 1.3$ for a standard $R_V = 3.1$ extinction law ([Fitzpatrick 1999](https://doi.org/10.1086/316293); [Wang & Chen 2019](https://doi.org/10.3847/1538-4357/ab15f2)). Agreement at this level confirms that the empirical period-color calibration is recovering the same physical reddening as SFD in low-to-moderate dust columns.

### Large-SFD Regime (SFD $>$ {LARGE_SFD_MIN} mag)

The {large_mask.sum()} stars in the large-SFD sample (median SFD = {large_median_sfd:.1f} mag) show a striking decoupling: the empirical E(G_{{BP}}-G_{{RP}}) remains near {large_median_empirical:.2f} mag while SFD values span the full range above 10 mag. Two physical effects combine to produce this apparent saturation:

1. **Distance-limited sampling.** RR Lyrae are point sources at fixed distances (~1–30 kpc). The SFD map integrates dust all the way to infinity. In the heavily obscured galactic plane, the bulk of the dust column lies beyond the star, so the stellar measurement records only a fraction of the SFD column.

2. **Gaia photometric quality cuts.** Stars in highly obscured sightlines tend to have poor BP/RP flux ratios (low SNR, crowded fields), and they fail the `phot_bp/rp_mean_flux_over_error > 5` criterion applied in nb07. The surviving sample is already biased toward foreground, moderately reddened stars.

The large-SFD regime therefore does not indicate calibration failure: it illustrates exactly the regime where a 2D total-column map (SFD) and a 3D distance-limited stellar measurement (this work) are expected to diverge, and where a true 3D dust map such as Bayestar ([Green et al. 2019](https://doi.org/10.3847/1538-4357/ab5362)) is required.
"""

display(Markdown(regime_discussion))

In [ ]:
from IPython.display import Markdown, display

r2_0_10  = lat_table['R2'][0]
r2_10_20 = lat_table['R2'][1]
r2_20_30 = lat_table['R2'][2]
r2_30_40 = lat_table['R2'][3]

discussion = rf"""
## Analysis and Discussion

The side-by-side maps show that the broad Galactic structure agrees well. Both maps become redder toward the Galactic plane and the inner Milky Way, which is exactly what one expects if both are tracing the large-scale dust distribution ([Schlegel, Finkbeiner, & Davis 1998](https://doi.org/10.1086/305772); [Schlafly & Finkbeiner 2011](https://doi.org/10.1088/0004-637X/737/2/103)). The matched-sightline density plot should therefore be read mainly as a monotonic trend check rather than as a one-to-one calibration.

### Physical Interpretation of 2D vs. 3D

SFD is a two-dimensional, total-column reddening map calibrated from IRAS and DIRBE far-infrared emission with a temperature correction derived from the 100 μm / 60 μm flux ratio ([Schlegel, Finkbeiner, & Davis 1998](https://doi.org/10.1086/305772)). By construction it integrates the full dust column from here to infinity, providing a single number per sightline independent of distance. Our RR Lyrae map, by contrast, measures the color excess only out to the finite distance of each individual pulsator, which traces the old stellar population at distances of order 1–30 kpc.

At high Galactic latitudes ($|b| \gtrsim 30°$) the thin-disk dust scale height ($\\approx 150$–200 pc) means that most of the SFD column is accumulated within the first few hundred parsecs of the disk, well inside the typical RR Lyrae distance. In that regime the two measurements should agree well in a statistical sense. Near the plane ($|b| \lesssim 15°$) the agreement degrades in physical interpretability even when $R^2$ stays high, because SFD always includes dust behind the star while the RR Lyrae map stops at the stellar distance, and because Gaia completeness drops sharply in the crowded low-latitude fields.

### What Makes SFD Work and Fail

SFD works because dust in thermal equilibrium re-emits absorbed starlight as a modified blackbody, and the integrated far-infrared intensity is proportional to the total column density of dust. The temperature correction stabilizes the column estimate against sightline-to-sightline dust temperature variation. However, several failure modes are known:

- **Complex plane sightlines**: When multiple dust components at different distances and temperatures pile up along a single line of sight, the single-temperature assumption breaks down and the inferred column can be biased.
- **CMB contamination at high latitude**: At $|b| \gtrsim 60°$ the SFD signal is very faint and can be contaminated by CMB fluctuations and zodiacal light residuals, leading to artificial structures in the high-latitude reddening map ([Peek & Graves 2010](https://doi.org/10.1088/0004-637X/719/1/415)).
- **Planck-based improvements**: The higher angular resolution and better-constrained dust temperature from Planck allow the dust column to be estimated more accurately in complex regions. Meisner & Finkbeiner (2015) produced an improved dust map by fitting two-component models to the Planck and IRAS/DIRBE data, reducing systematic offsets in the SFD calibration.

### Interpretation of the Latitude-Dependent $R^2$ Table

The latitude-binned $R^2$ table should be interpreted empirically rather than from a fixed prior expectation. In the current notebook output, $R^2$ is actually strongest in the lowest-latitude bins: **$R^2 = {r2_0_10:.3f}$ for $0$–$10^\circ$** and **$R^2 = {r2_10_20:.3f}$ for $10$–$20^\circ$**, then it declines through the intermediate bins such as **$R^2 = {r2_20_30:.3f}$ for $20$–$30^\circ$** and **$R^2 = {r2_30_40:.3f}$ for $30$–$40^\circ$**. That does not mean the plane comparison is cleaner in an absolute sense. It means the plane has the largest dynamic range in dust, so $R^2$ can remain very high even though the two maps are not tracing the same total column to the same distance. High-latitude bins have a much smaller reddening range, so even modest noise or depth mismatch can reduce $R^2$ there.

The right physical conclusion is therefore two-part. First, the broad monotonic ordering of dusty versus clean sightlines agrees well enough that the RR Lyrae map is clearly recovering the same large-scale Galactic dust structure as SFD. Second, one should **not** expect exact map equality, especially in the plane, because the two products encode different observables: SFD is a 2D total-column map, while the RR Lyrae map is a distance-limited stellar reddening measurement.

### References

- Green, G. M., Schlafly, E. F., Zucker, C., et al. (2019), *A 3D Dust Map Based on Gaia, Pan-STARRS 1, and 2MASS*, ApJ, 887, 93. https://doi.org/10.3847/1538-4357/ab5362
- Meisner, A. M., & Finkbeiner, D. P. (2015), *The two-component dust spectral energy distribution of Planck and IRAS*, ApJ, 798, 88. https://doi.org/10.1088/0004-637X/798/2/88
- Peek, J. E. G., & Graves, G. J. (2010), *A CMB contamination analysis of the Schlegel et al. Galactic extinction map*, ApJ, 719, 415. https://doi.org/10.1088/0004-637X/719/1/415
- Schlegel, D. J., Finkbeiner, D. P., & Davis, M. (1998), *Maps of Dust Infrared Emission for Use in Estimation of Reddening and Cosmic Microwave Background Radiation Foregrounds*, ApJ, 500, 525. https://doi.org/10.1086/305772
- Schlafly, E. F., & Finkbeiner, D. P. (2011), *Measuring Reddening with Sloan Digital Sky Survey Stellar Spectra and Recalibrating SFD*, ApJ, 737, 103. https://doi.org/10.1088/0004-637X/737/2/103
"""

display(Markdown(discussion))

In [ ]:
from ugdatalab import plot_aitoff_reddening_dark

all_mask = np.ones(len(rrlyrae_clean), dtype=bool)
dark_vmax = float(np.nanpercentile(
    np.asarray(rrlyrae_clean['E_bprp'], dtype=float), 99.5
))

fig_dark, ax_dark = plot_aitoff_reddening_dark(
    rrlyrae_clean,
    all_mask,
    vmin=0.0,
    vmax=dark_vmax,
    save=True,
)
plt.show()